In [ ]:
!pip install transformers datasets librosa torch

In [ ]:
import torch
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt
import os
import time

from transformers import AutoModelForAudioClassification, AutoFeatureExtractor
from datasets import load_dataset, Audio, DatasetDict
from scipy.interpolate import interp1d

In [ ]:
TARGET_IDX = 35

### Φόρτωση finetuned μοντέλου

In [ ]:
repo_id = "VasilisAsim/whisper-finetuned-TESS"
model_ft = AutoModelForAudioClassification.from_pretrained(repo_id)
feature_extractor = AutoFeatureExtractor.from_pretrained(repo_id)

device = "cuda" if torch.cuda.is_available() else "cpu"
model_ft = model_ft.to(device)
model_ft.eval()

In [ ]:
label2id = model_ft.config.label2id
id2label = model_ft.config.id2label

### Φόρτωση dataset (speaker-independent)

In [ ]:
from datasets import load_dataset, Audio, DatasetDict

def load_tess_speaker_independent():
    dataset = load_dataset("AbstractTTS/TESS", split="train")
    dataset = dataset.cast_column("audio", Audio(decode=False))

    def get_speaker(example):
        filename = example["audio"]["path"]
        if "OAF" in filename:
            return "OAF"
        elif "YAF" in filename:
            return "YAF"
        return "unknown"

    train_dataset = dataset.filter(lambda x: get_speaker(x) == "OAF")
    test_dataset = dataset.filter(lambda x: get_speaker(x) == "YAF")

    train_dataset = train_dataset.cast_column("audio", Audio(sampling_rate=16_000))
    test_dataset = test_dataset.cast_column("audio", Audio(sampling_rate=16_000))

    return DatasetDict({"train": train_dataset, "test": test_dataset})

In [ ]:
dataset = load_tess_speaker_independent()

### Wrapper για forward pass

In [ ]:
"""
Wrapper που επιστρέφει μόνο τα logits του μοντέλου, ώστε forward hooks/
backward hooks να καταγράψουν activations και gradients στο
τελευταίο transformer layer χωρίς να χρειάζεται να ξαναγράφεται η κλήση
του μοντέλου σε κάθε μέθοδο.
"""
class Wrapper(torch.nn.Module):
    def __init__(self, model):
        super().__init__()
        self.model = model
    def forward(self, x):
        return self.model(x).logits

In [ ]:
wrapped_model = Wrapper(model_ft).to(device).eval()

### Βοηθητικές συναρτήσεις

In [ ]:
def load_sample(idx, split="test"):
    sample = dataset[split][idx]
    audio = sample["audio"]
    true_label = sample["emotion"]

    input_values = feature_extractor(
        audio["array"], sampling_rate=feature_extractor.sampling_rate, return_tensors="pt"
    ).input_features.to(device)

    return input_values, true_label, audio["array"]

In [ ]:
input_values, true_label, speech = load_sample(TARGET_IDX, split="test")
print(f"True label: {true_label}")

### Καταγραφή activations/gradients τελευταίου layer

In [ ]:
#Επιλογή target layer (τελευταίο transformer block)
target_layer = model_ft.encoder.layers[-1]

acts, grads = {}, {}

def fwd_hook(m, i, o):
    acts['A'] = o[0] if isinstance(o, tuple) else o

def bwd_hook(m, gi, go):
    grads['dA'] = go[0]

h1 = target_layer.register_forward_hook(fwd_hook)
h2 = target_layer.register_full_backward_hook(bwd_hook)

In [ ]:
#Forward + backward
logits = wrapped_model(input_values)
class_idx = logits.argmax(-1).item()
model_ft.zero_grad()
logits[0, class_idx].backward()

A  = acts['A'][0].detach()
dA = grads['dA'][0].detach()

def norm(x):
    x = x - x.min()
    return x / (x.max() + 1e-8)

### Υπολογισμός Grad-CAM, Grad-CAM++, Eigen-CAM, Score-CAM, Ablation-CAM

In [ ]:
#Grad-CAM
w = dA.mean(dim=0)
cam_gradcam = norm(F.relu((w * A).sum(dim=-1))).cpu().numpy()

#Grad-CAM++
g2, g3 = dA**2, dA**3
denom = 2*g2 + A.sum(0, keepdim=True)*g3
denom = torch.where(denom != 0, denom, torch.ones_like(denom))
alpha = g2 / denom
w = (alpha * F.relu(dA)).sum(dim=0)
cam_gradcampp = norm(F.relu((w * A).sum(dim=-1))).cpu().numpy()

#Eigen-CAM
A_np = (A - A.mean(0, keepdim=True)).cpu().numpy()
_, _, Vt = np.linalg.svd(A_np, full_matrices=False)
cam_eigen = np.abs(A_np @ Vt[0])
cam_eigen = cam_eigen / (cam_eigen.max() + 1e-8)

#Score-CAM
T, C = A.shape
up = F.interpolate(A.T.unsqueeze(0), size=input_values.shape[-1], mode='linear')[0]  # (C, 3000)
up = (up - up.min(1, keepdim=True).values) / (up.max(1, keepdim=True).values - up.min(1, keepdim=True).values + 1e-8)

scored_weights = []
with torch.no_grad():
    for i in range(0, C, 32):
        mask_batch = up[i:i+32].unsqueeze(1).to(device)   # (batch_c, 1, 3000)
        masked = mask_batch * input_values                 # (batch_c, 80, 3000)
        out = F.softmax(wrapped_model(masked), dim=-1)[:, class_idx]
        scored_weights.append(out.cpu())
w = torch.cat(scored_weights)
cam_score = norm(F.relu((w.unsqueeze(0) * A.cpu()).sum(-1))).numpy()

#Ablation-CAM
with torch.no_grad():
    base = F.softmax(wrapped_model(input_values), -1)[0, class_idx].item()

abl_weights = torch.zeros(C)
for c in range(C):
    def hook(m, i, o, c=c):
        o2 = (o[0] if isinstance(o, tuple) else o).clone()
        o2[..., c] = 0
        return (o2,) + o[1:] if isinstance(o, tuple) else o2
    hh = target_layer.register_forward_hook(hook)
    with torch.no_grad():
        s = F.softmax(wrapped_model(input_values), -1)[0, class_idx].item()
    hh.remove()
    abl_weights[c] = (base - s) / (base + 1e-8)

cam_ablation = norm(F.relu((abl_weights.unsqueeze(0) * A.cpu()).sum(-1))).numpy()

h1.remove(); h2.remove()

### Οπτικοποίηση και αποθήκευση attributions

In [ ]:
import time

n_samples = input_values.shape[-1]
def upsample(c):
    return F.interpolate(torch.tensor(c).view(1,1,-1), size=n_samples, mode='linear').squeeze().numpy()

cams = {
    "Grad-CAM": cam_gradcam,
    "Grad-CAM++": cam_gradcampp,
    "Eigen-CAM": cam_eigen,
    "Score-CAM": cam_score,
    "Ablation-CAM": cam_ablation,
}

wav = speech
fig, axes = plt.subplots(len(cams)+1, 1, figsize=(11, 1.6*(len(cams)+1)), sharex=True)
axes[0].plot(wav, color="steelblue", linewidth=0.6)
axes[0].set_title("Κυματομορφή εισόδου")

for ax, (name, c) in zip(axes[1:], cams.items()):
    ax.plot(wav, color="lightgray", linewidth=0.5)
    ax.imshow(upsample(c)[np.newaxis, :], cmap="jet", aspect="auto",
              extent=[0, len(wav), wav.min(), wav.max()], alpha=0.55)
    ax.set_ylabel(name, rotation=0, ha="right", va="center", fontsize=9)

axes[-1].set_xlabel("δείγμα (sample)")
fig.suptitle(f"Πρόβλεψη κλάσης: {model_ft.config.id2label[class_idx]}", fontsize=12)
plt.tight_layout()
plt.show()